# 02 — Model detective (Student workbook)

**Time:** 70 minutes
**Question:** when does the dataset contain enough information to distinguish
a sphere from an ellipsoid?

The file name deliberately reveals nothing about the generating model. Split
the work: one partner advocates for a sphere, the other for an ellipsoid.

> **Lecture connection**
>
> - Slide 24, *Welcome to the SANS zoo*: many shapes share a plateau and a
>   decay and differ in oscillation depth and mid-q slope. Browse the
>   [SasView model marketplace](https://marketplace.sasview.org/) for the full zoo.
> - Slide 25, *Name that shape!*: panels B and C "look alike over a narrow
>   Q-window", and an extended Q-range shows the difference. This notebook is
>   that statement made quantitative.
> - Slide 21, *Guinier analysis*: the $R_g$ table. Every shape looks the same at
>   low Q apart from $I(0)$ and $R_g$.
> - Slide 34, *Deeper into uncertainty*: "choosing a model is also a prior. When
>   two models fit equally well, physics and sample knowledge decide, not χ²."
>   Bring your figures from this notebook to the plenary discussion.
> - Slide 50, *Polymers*: a sphere, a coil and a rod with the *same* $R_g$ cannot
>   be told apart at low Q. Guinier only sees size; the shape lives at higher Q.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'mystery_shape.csv'

# Load the data and draw a candidate model, with its starting values, on top.
# Nothing is fitted yet: plot_model() only shows the model as it stands.
preview = SANSFitter()
preview.load_data(str(filename))
preview.set_model('sphere')
preview.set_param('sld', value=4.2)
preview.set_param('sld_solvent', value=1.0)
preview.set_param('radius', value=45)
preview.set_param('scale', value=0.04)
preview.set_param('background', value=0.002)
fig = preview.plot_model()


### Look before you fit

The lecture's fitting strategy (slide 32) starts with reading the data, not
with the optimizer. `plot_model()` draws the data with a candidate model, here
a 45 Å sphere, the starting point of the sphere fit below. Change the
`radius` value in that cell and rerun it: can any single sphere radius match
both the plateau and the decay? Then compare the data with
`sphere_reference.csv` from notebook 01:

- Are the form-factor minima as deep and sharp? Shallow, washed-out minima
  can mean polydispersity or smearing, but also **anisotropy**: an
  orientationally averaged ellipsoid has a different dimension in every
  direction, so its minima fall at different q and blur together.
- Where does the plateau bend over, and what slope follows it (slide 22)?
  A very long object would show an extended $Q^{-1}$ region. A moderately
  elongated one shows only a hint of it.
- Notice that these data are noisier than in notebook 01. How might that
  affect your ability to tell two models apart?

## Exercise 1 — Competing explanations

The first cell repeats the notebook-01 workflow for a sphere, wrapped in a
function so it can be rerun later on part of the q range. The SLDs stay fixed;
only radius, scale and background vary. Compare residual patterns, not just
the numerical objective.

In [ ]:
def fit_sphere(qmin=None, qmax=None):
    # The notebook-01 workflow wrapped in a function, so it can be rerun
    # on a different q range. qmin and qmax are in Å^-1; None means "use all data".
    fitter = SANSFitter()
    fitter.load_data(str(filename))
    if qmin is not None or qmax is not None:
        fitter.set_q_range(qmin=qmin, qmax=qmax)
    fitter.set_model('sphere')
    # Same contrast as before, so the SLDs stay fixed.
    fitter.set_param('sld', value=4.2, vary=False)
    fitter.set_param('sld_solvent', value=1.0, vary=False)
    # Broad bounds: we do not yet know whether a sphere is even the right shape.
    fitter.set_param('radius', value=45, min=10, max=180, vary=True)
    fitter.set_param('scale', value=0.04, min=0.001, max=0.2, vary=True)
    fitter.set_param('background', value=0.002, min=0, max=0.02, vary=True)
    fitter.fit(engine='lmfit', method='least_squares')
    return fitter


# Fit the full q range and show the residuals.
sphere = fit_sphere()
sphere.plot_results(show_residuals=True, log_scale=True)


### A competing model

Set up the ellipsoid with its own `SANSFitter`, so the two fits stay
independent. After `set_model('ellipsoid')`, `get_params()` shows its two size
parameters. Decide which parameters to fix and which to vary. Hint: if both
radii start at the same value, the optimizer begins from a sphere; is that a
fair start for testing an elongated particle?

In [ ]:
# A new, separate fitter, so the sphere fit above is left untouched.
ellipsoid = SANSFitter()
ellipsoid.load_data(str(filename))

# YOUR CODE HERE, following the same steps as for the sphere:
#   1. ellipsoid.set_model('ellipsoid'), then ellipsoid.get_params() to see
#      its parameters (it has radius_polar and radius_equatorial).
#   2. set_param lines: fix both SLDs as before; give the two radii, scale and
#      background starting values, bounds and vary=True.
#   3. ellipsoid.fit(engine='lmfit', method='least_squares')
#   4. ellipsoid.plot_results(show_residuals=True, log_scale=True)
# Keep the name `ellipsoid`: later cells use it.
# Delete the `raise` line below when your lines are in place.
raise NotImplementedError('Configure and fit the ellipsoid, then delete this line')


In [ ]:
# The two fit reports, one above the other. Compare the reduced χ², the
# residual checks, and whether the fitted sizes are physically reasonable.
print('SPHERE')
print(sphere.get_fit_report())
print()
print('ELLIPSOID')
print(ellipsoid.get_fit_report())


## Exercise 2 — Remove information

Repeat both fits using only `0.004 ≤ q ≤ 0.025 Å⁻¹`. The sphere fit is done
for you; set up the ellipsoid fit yourself. Predict what will happen before
running the cells. Which shape features have disappeared from view?

*Before you run:* convert the window to real space with $d \approx 2\pi/Q$,
as in notebook 00. What is the smallest distance still resolved, and how does
it compare with the particle dimensions you just fitted?

In [ ]:
# Sphere: reuse fit_sphere from Exercise 1, now with only 0.004 <= q <= 0.025 Å^-1.
sphere_short = fit_sphere(qmin=0.004, qmax=0.025)
sphere_short.plot_results(show_residuals=True, log_scale=True)
print(sphere_short.get_fit_report())


In [ ]:
# Ellipsoid on the restricted window.
ellipsoid_short = SANSFitter()
ellipsoid_short.load_data(str(filename))
ellipsoid_short.set_q_range(qmin=0.004, qmax=0.025)

# YOUR CODE HERE: repeat your ellipsoid setup from Exercise 1 for
# `ellipsoid_short` (set_model, set_param lines), then
#   ellipsoid_short.fit(engine='lmfit', method='least_squares')
#   ellipsoid_short.plot_results(show_residuals=True, log_scale=True)
#   print(ellipsoid_short.get_fit_report())
# Keep the name `ellipsoid_short`: the next cells use it.
# Delete the `raise` line below when your lines are in place.
raise NotImplementedError('Fit the ellipsoid on the restricted range, then delete this line')


### Why the models become indistinguishable

The lecture called the Guinier law *universal*: at low Q every shape gives
$I(Q) \approx I(0)\exp(-Q^2R_g^2/3)$, which is valid for $QR_g \lesssim 1.3$
(slide 21). In that region the data carry essentially **two numbers**, $I(0)$
and $R_g$. Any model that can reproduce those two numbers fits equally well.

The cell below tests this with your fitted parameters, using
$R_g^2 = (2R_e^2 + R_p^2)/5$ for a spheroid with equatorial radius $R_e$ and
polar radius $R_p$, and $R_g^2 = \tfrac{3}{5}R^2$ for a sphere. Look at the
printed numbers and answer:

1. How close are the two restricted-range $R_g$ values?
2. Where does $Q_{\max}R_g$ sit relative to the Guinier limit of 1.3?
3. Is the equal-$R_g$ sphere radius close to the radius the restricted sphere
   fit found?

This is **non-identifiability**: the data are compatible with a family of
parameter sets, so the optimizer's single answer is not a measurement of
shape. Notebook 04 shows the same idea as correlated posterior distributions.

In [ ]:
def value(fitter, name):
    # The fitted number for one parameter, read from the fit report.
    return fitter.get_fit_report().parameters[name]['value']


def rg_spheroid(r_equatorial, r_polar):
    return np.sqrt((2 * r_equatorial**2 + r_polar**2) / 5)


def rg_sphere(radius):
    return np.sqrt(3 / 5) * radius


rg_sphere_short = rg_sphere(value(sphere_short, 'radius'))
rg_ellipsoid_short = rg_spheroid(
    value(ellipsoid_short, 'radius_equatorial'),
    value(ellipsoid_short, 'radius_polar'),
)
rg_ellipsoid_full = rg_spheroid(
    value(ellipsoid, 'radius_equatorial'), value(ellipsoid, 'radius_polar')
)

print(f'1. Restricted range: Rg(sphere) = {rg_sphere_short:.1f} Å, '
      f'Rg(ellipsoid) = {rg_ellipsoid_short:.1f} Å')
print(f'2. Qmax * Rg = 0.025 * {rg_ellipsoid_full:.1f} = {0.025 * rg_ellipsoid_full:.2f}')
print(f'3. Sphere with the full-range ellipsoid Rg: R = {np.sqrt(5 / 3) * rg_ellipsoid_full:.1f} Å; '
      f'restricted sphere fit: R = {value(sphere_short, "radius"):.1f} Å')


## Verdict

Write a short referee-style verdict. Include one reason based on residuals, one
based on parameter plausibility, and one follow-up measurement that would make
the distinction stronger.

Follow-up measurements from the lecture to consider:

- **Extend q.** A shorter detector distance or a shorter wavelength raises
  $Q_{\max}$ (slides 9 and 39–41). Ask which q range the shape information
  lives in.
- **Change the contrast.** H/D variation (slides 14 and 16) changes $\Delta\rho$
  and so constrains the scale–contrast degeneracy, but does contrast alone
  change the *shape* of the curve for a homogeneous particle?
- **Complementary techniques.** Slide 7 asked when SANS and TEM would disagree
  about a size. Here, what could a micrograph settle that SANS cannot, and
  what could it get wrong?
- **Field or flow alignment.** Anisotropic objects give anisotropic 2D
  patterns when aligned (compare slide 29): a radially averaged 1D curve has
  thrown that information away.

**Your answer** *(double-click this cell to edit it, then press Shift + Enter)*

Write your referee-style verdict: one reason from the residuals, one from parameter plausibility, and one follow-up measurement.

## Further reading

- [SasView `ellipsoid` model](https://www.sasview.org/docs/user/models/ellipsoid.html): parameter
  definitions. `radius_polar` lies along the rotation axis, so $R_p > R_e$ is
  prolate (cigar-like) and $R_p < R_e$ is oblate (disc-like).
- [Jacques & Trewhella, *Small-angle scattering for structural biology: expanding the frontier while avoiding the pitfalls*, Protein Sci. 19, 642 (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2867006/): the discussion of model ambiguity and of reporting what the data can
  and cannot support.